# **IMPORT LIBRARIES**

In [123]:
# Import
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
import warnings
warnings.filterwarnings('ignore')

# **DATA LOADING**

In [124]:
df = pd.read_csv('data_nilai_peserta.csv')
display(df.head(5))
display(df.tail(5))
df.info()

,id_user,nilai_biologi,nilai_fisika,nilai_kimia,nilai_matematika,nilai_kmb,nilai_kpu,nilai_kua,nilai_ppu,jurusan_tujuan,universitas_tujuan,kategori_jurusan
0,4,400,400,400,400,400,400,400,400,SEKOLAH ILMU & TEKNOLOGI HAYATI - PROGRAM SAINS,INSTITUT TEKNOLOGI BANDUNG,Science
1,14,816,666,651,695,678,685,706,562,PENDIDIKAN DOKTER,UNIVERSITAS INDONESIA,Kesehatan
2,19,562,839,624,551,700,781,464,668,ILMU DAN TEKNOLOGI PANGAN,UNIVERSITAS BRAWIJAYA,Science
3,23,700,669,692,507,679,692,813,573,FAKULTAS TEKNIK MESIN & DIRGANTARA (FTMD),INSTITUT TEKNOLOGI BANDUNG,Teknik
4,28,461,619,441,666,593,563,500,370,TEKNIK PERANGKAT LUNAK (TEKNOLOGI INFORMATIKA),UNIVERSITAS PALANGKARAYA,Teknologi


,id_user,nilai_biologi,nilai_fisika,nilai_kimia,nilai_matematika,nilai_kmb,nilai_kpu,nilai_kua,nilai_ppu,jurusan_tujuan,universitas_tujuan,kategori_jurusan
86564,344111,516,412,376,549,546,523,490,520,TEKNIK LINGKUNGAN,UNIVERSITAS ANDALAS,Teknik
86565,344125,334,634,435,399,613,461,390,554,FARMASI,UNIVERSITAS NEGERI SEMARANG,Kesehatan
86566,344127,316,717,421,327,474,494,483,554,PENDIDIKAN FISIKA,UNIVERSITAS SEBELAS MARET,Pendidikan
86567,344151,592,451,583,508,437,447,485,492,AGRIBISNIS,UNIVERSITAS MALIKUSSALEH,Bisnis
86568,344192,411,602,415,494,519,540,445,547,AGRONOMI,UNIVERSITAS LAMPUNG,Science


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 86569 entries, 0 to 86568
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   id_user             86569 non-null  int64 
 1   nilai_biologi       86569 non-null  int64 
 2   nilai_fisika        86569 non-null  int64 
 3   nilai_kimia         86569 non-null  int64 
 4   nilai_matematika    86569 non-null  int64 
 5   nilai_kmb           86569 non-null  int64 
 6   nilai_kpu           86569 non-null  int64 
 7   nilai_kua           86569 non-null  int64 
 8   nilai_ppu           86569 non-null  int64 
 9   jurusan_tujuan      86569 non-null  object
 10  universitas_tujuan  86569 non-null  object
 11  kategori_jurusan    86569 non-null  object
dtypes: int64(9), object(3)
memory usage: 7.9+ MB


In [125]:
nilai_cols = [
    'nilai_biologi', 'nilai_fisika', 'nilai_kimia', 'nilai_matematika',
    'nilai_kmb', 'nilai_kpu', 'nilai_kua', 'nilai_ppu'
]

# **MODELLING**

In [126]:
def rekomendasi_knn(nilai_siswa, nama='Siswa', k=100, top_n=10):
    """
    Pure KNN recommendation:
    1. Hitung jarak ke semua siswa (via NearestNeighbors)
    2. Ambil k tetangga terdekat
    3. Ranking jurusan berdasarkan frekuensi
    """
    vals = np.array(nilai_siswa).reshape(1, -1)
    vals_scaled = scaler.transform(vals)

    # Cari tetangga
    distances, indices = nn.kneighbors(vals_scaled)
    neighbors = df.iloc[indices[0]].copy()
    neighbors['_distance'] = distances[0]

    # Ranking jurusan
    jurusan_rank = neighbors['jurusan_tujuan'].value_counts().head(top_n)
    total = len(neighbors)

    # Distribusi kategori
    kategori_dist = neighbors['kategori_jurusan'].value_counts()
    kategori_dominan = kategori_dist.index[0]
    kategori_pct = kategori_dist.iloc[0] / total * 100

    # Top kategori
    top_kategori = [(cat, round(cnt / total * 100, 1))
                    for cat, cnt in kategori_dist.head(4).items()]

    return {
        'nama': nama,
        'nilai': nilai_siswa,
        'kategori_dominan': kategori_dominan,
        'prob_kategori': round(kategori_pct, 1),
        'top_kategori': top_kategori,
        'rekomendasi': [{'jurusan': j, 'jumlah': int(cnt), 'persen': round(cnt/total*100, 1)}
                       for j, cnt in jurusan_rank.items()]
    }

def tampilkan(hasil):
    print('=' * 65)
    print(f'  REKOMENDASI -- {hasil["nama"].upper()}')
    print('=' * 65)

    cols = ['Bio', 'Fis', 'Kim', 'Mat', 'KMB', 'KPU', 'KUA', 'PPU']
    print(f'  Nilai: {" | ".join(f"{c}: {v}" for c,v in zip(cols, hasil["nilai"]))}')
    print()

    print(f'  Top Kategori:')
    for k, p in hasil['top_kategori']:
        marker = '>>' if k == hasil['kategori_dominan'] else '  '
        print(f'    {marker} {k:15s} {p:5.1f}%')
    print()

    print(f'  REKOMENDASI JURUSAN:')
    for rank, r in enumerate(hasil['rekomendasi'], start=1):
        print(f'    {rank:>2}. {r["jurusan"][:45]:45s} {r["jumlah"]:>3} siswa ({r["persen"]:.1f}%)')

    return hasil

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[nilai_cols].values)

K = 100
nn = NearestNeighbors(n_neighbors=K, metric='euclidean', n_jobs=-1)
nn.fit(X_scaled)

Model KNN siap -- akan mencari 100 tetangga terdekat dari 86,569 siswa.


# **EVALUASI**

In [ ]:
np.random.seed(42)
n_eval = 500  
eval_idx = np.random.choice(len(df), size=n_eval, replace=False)

distances_all, indices_all = nn.kneighbors(X_scaled[eval_idx], n_neighbors=K + 1)

hit_top1 = 0
hit_top3 = 0

for row_i, idx in enumerate(eval_idx):
    neighbor_idx = indices_all[row_i]
    neighbor_idx = neighbor_idx[neighbor_idx != idx][:K]
    neighbors = df.iloc[neighbor_idx]

    kategori_dist = neighbors['kategori_jurusan'].value_counts()
    top1 = kategori_dist.index[0]
    top3 = kategori_dist.index[:3].tolist()

    actual = df.iloc[idx]['kategori_jurusan']
    if actual == top1:
        hit_top1 += 1
    if actual in top3:
        hit_top3 += 1

acc_top1 = hit_top1 / n_eval * 100
acc_top3 = hit_top3 / n_eval * 100

print(f'Leave-One-Out Evaluation (n={n_eval} siswa, K={K}):')
print(f'  Top-1 Kategori Hit Rate : {acc_top1:.1f}%')
print(f'  Top-3 Kategori Hit Rate : {acc_top3:.1f}%')


Leave-One-Out Evaluation (n=500 siswa, K=100):
  Top-1 Kategori Hit Rate : 28.4%
  Top-3 Kategori Hit Rate : 80.2%


In [ ]:
profil_uji = [550, 820, 790, 850, 780, 620, 810, 770]  # contoh profil siswa untuk uji sensitivitas K

for k in [25, 50, 100, 200]:
    h = rekomendasi_knn(profil_uji, 'Uji-K', k=k, top_n=3)
    j1 = h['rekomendasi'][0]['jurusan'][:40]
    j2 = h['rekomendasi'][1]['jurusan'][:40] if len(h['rekomendasi']) > 1 else ''
    print(f'  K={k:>3}: {j1:40s} {j2}')


  K= 25: PENDIDIKAN DOKTER                        SEKOLAH TEKNOLOGI ELEKTRO & INFORMATIKA 
  K= 50: PENDIDIKAN DOKTER                        SEKOLAH TEKNOLOGI ELEKTRO & INFORMATIKA 
  K=100: PENDIDIKAN DOKTER                        SEKOLAH TEKNOLOGI ELEKTRO & INFORMATIKA 
  K=200: PENDIDIKAN DOKTER                        SEKOLAH TEKNOLOGI ELEKTRO & INFORMATIKA 

Semakin besar K, rekomendasi cenderung lebih stabil/general; semakin kecil K, lebih spesifik namun lebih rentan noise.


In [130]:
samples = [
    ('Andi',   [800, 820, 790, 850, 780, 800, 810, 770]),
    ('Budi',   [650, 720, 680, 710, 640, 670, 700, 660]),
    ('Citra',  [500, 480, 510, 520, 490, 500, 480, 510]),
    ('Dedi',   [350, 380, 340, 370, 360, 350, 340, 330]),
    ('Euis',   [900, 420, 450, 950, 520, 500, 630, 510]),
]

all_hasil = []
for name, values in samples:
    h = rekomendasi_knn(values, name)
    tampilkan(h)
    print()
    all_hasil.append(h)


  REKOMENDASI -- ANDI
  Nilai: Bio: 800 | Fis: 820 | Kim: 790 | Mat: 850 | KMB: 780 | KPU: 800 | KUA: 810 | PPU: 770

  Top Kategori:
    >> Kesehatan        44.0%
       Teknik           23.0%
       Teknologi        14.0%
       Science          13.0%

  REKOMENDASI JURUSAN:
     1. PENDIDIKAN DOKTER                              27 siswa (27.0%)
     2. KEDOKTERAN                                     10 siswa (10.0%)
     3. SEKOLAH TEKNOLOGI ELEKTRO & INFORMATIKA (STEI   9 siswa (9.0%)
     4. FAKULTAS TEKNIK MESIN & DIRGANTARA (FTMD)       5 siswa (5.0%)
     5. MATEMATIKA                                      5 siswa (5.0%)
     6. FAKULTAS TEKNOLOGI INDUSTRI (FTI) - KAMPUS GA   5 siswa (5.0%)
     7. FARMASI                                         4 siswa (4.0%)
     8. FAKULTAS TEKNIK SIPIL & LINGKUNGAN (FTSL) - K   3 siswa (3.0%)
     9. AGRIBISNIS                                      3 siswa (3.0%)
    10. FAKULTAS MATEMATIKA & ILMU PENGET. ALAM (FMIP   2 siswa (2.0%)

  REKOMEN

In [ ]:
print('=' * 120)
print('   TABEL PERBANDINGAN 5 SISWA')
print('=' * 120)
print(f'{"Nama":10s} {"Bio":>5s} {"Fis":>5s} {"Kim":>5s} {"Mat":>5s} Kategori     Jurusan #1                   Top 2')
print('-' * 120)
for h in all_hasil:
    n = h['nilai']
    r = h['rekomendasi']
    j1 = r[0]['jurusan'][:35] if r else '-'
    j2 = r[1]['jurusan'][:25] if len(r) > 1 else ''
    print(f'{h["nama"]:10s} {n[0]:>5d} {n[1]:>5d} {n[2]:>5d} {n[3]:>5d} {h["kategori_dominan"]:12s} {j1:35s} {j2}')
print('=' * 120)


   TABEL PERBANDINGAN 5 SISWA
Nama         Bio   Fis   Kim   Mat Kategori     Jurusan #1                   Top 2
------------------------------------------------------------------------------------------------------------------------
Andi         800   820   790   850 Kesehatan    PENDIDIKAN DOKTER                   KEDOKTERAN
Budi         650   720   680   710 Teknik       PENDIDIKAN DOKTER                   KEDOKTERAN
Citra        500   480   510   520 Teknik       KIMIA                               TEKNIK MESIN
Dedi         350   380   340   370 Teknik       TEKNIK ELEKTRO                      TEKNIK INFORMATIKA
Euis         900   420   450   950 Science      PENDIDIKAN DOKTER                   BIOLOGI


In [ ]:
pipeline_artefacts = {
    'scaler': scaler,
    'knn_model': nn,
    'nilai_cols': nilai_cols,
    'dataset_lengkap': df
}

os.makedirs('model', exist_ok=True)
file_path = 'your_major_recomendation_pipeline.pkl'
joblib.dump(pipeline_artefacts, file_path)

size = os.path.getsize(file_path) / 1024 / 1024
print(f'Model berhasil tersimpan dalam 1 file pipeline:')
print(f'    {file_path} ({size:.1f} MB)')

Model berhasil tersimpan dalam 1 file pipeline:
    your_major_recomendation_pipeline.pkl (18.9 MB)
